In [1]:
import sys
sys.path.insert(0, '..')



In [ ]:
from utils.helpers import create_aliases_patterns_map
from utils.files import read_cached_array


In [ ]:
from utils.files import read_cached_array
aliases = read_cached_array("../data/minimized/aliases.pkl")
aliases_patterns_map = create_aliases_patterns_map(aliases)

In [2]:
from collections import defaultdict
from utils.files import read_cached_array

triples = read_cached_array("../data/minimized/triples_train.pkl")
descriptions = read_cached_array("../data/minimized/descriptions.pkl")
relations = read_cached_array("../data/minimized/relations.pkl")
golden_triples= read_cached_array("../data/minimized/golden_triples.pkl")


triples_per_head = defaultdict(list)
for h,r,t in triples:
    triples_per_head[h].append((r,t))

In [ ]:
from transformers import BertTokenizerFast
tokenizer= BertTokenizerFast.from_pretrained('bert-base-uncased')



In [ ]:
d_ids = list(descriptions.keys())
triple_ids = set([h for h,r,t in triples])
tails_ids = set([t for h,r,t in triples])
aliases_should_be = triple_ids.union(tails_ids)
aliases_ids = set(aliases.keys())

len(d_ids), len(triple_ids), len(tails_ids), len(aliases_ids), len(aliases_should_be)


In [ ]:
sentences = list(descriptions.values())
enc = tokenizer(sentences, return_offsets_mapping=False, add_special_tokens=False, return_attention_mask=False, return_token_type_ids=False, padding="max_length", truncation=True, max_length=128)

In [ ]:
tt = ['what', 'it', 'takes', ':', 'the', 'way', 'to', 'the', 'white', 'house', '(', 'isbn', '978', '##0', '##39', '##45', '##6', '##26', '##0', '##5', ')', 'is', 'a', 'book', 'about', 'the', '1988', 'presidential', 'election', 'in', 'the', 'united', 'states', 'written', 'by', 'richard', 'ben', 'cramer', '.', 'it', 'was', 'published', 'in', '1992', '.', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]', '[PAD]'] 
tt[25]

In [ ]:
golden_triples["Q303824"]

[((1, 7), 'P31', (89, 89)), ((60, 66), 'P31', (89, 89))]

: 

In [ ]:
dd = "What it Takes: The Way to the White House (iSBN 9780394562605) is a book about the 1988 presidential election in the United States written by Richard Ben Cramer. it was published in 1992."


enc2 = tokenizer(
    [dd],
    return_offsets_mapping=False, 
    add_special_tokens=False, 
    return_attention_mask=False, 
    return_token_type_ids=False,
    padding="max_length",
    truncation=True,
    max_length=128,
)


enc2.encodings[0].tokens[25]

In [ ]:
d_id = "Q16170662"
d_idx = d_ids.index(d_id)
enc.encodings[d_idx].tokens

In [ ]:
id = "Q135663"
_triples = triples_per_head[id]
_gold_triples = golden_triples[id]
_d_text = descriptions[id]
_d_idx = list(descriptions.keys()).index(id)
_d_idx

head_spans_found = [(9,12), (0,6)]

related_triples = [("P171", "Q266"), ("P141", "Q211005")]

not_found_triples = defaultdict(list)
tail_spans_cache = {}
for r, t in related_triples:
    tail_spans = set()
    if t not in tail_spans_cache:
        for pattern, als_str in [(aliases_patterns_map[als_str], als_str) for als_str in aliases[t]]:
            # print(f"searching for pattern: {pattern}")
            for match in pattern.finditer(_d_text):
                char_start, char_end = match.span()
                if char_end == 0 or char_start >= char_end:
                    continue
                ts = enc.char_to_token(_d_idx, char_start)
                te = enc.char_to_token(_d_idx, char_end - 1)
                if ts is not None and te is not None:
                    tail_spans.add((ts, te))
        tail_spans_cache[t] = tail_spans
    if len(tail_spans_cache[t]) == 0:
        not_found_triples[h].append((r, t))
        continue
print(f"tail_spans_cache: {tail_spans_cache}")
print(f"not_found_triples: {not_found_triples} ")
("P141", "Q211005") in not_found_triples[id]




In [ ]:
ll = [("a1", "r1", "a2" ) , ("a1", "r2", "a3" ) , ("a1", "r3", "a4")]
g = [("a1", "r1", "a2"), ("a1", "r")]

In [ ]:
triples_by_head = {}
h = 'Q20723'
related_triples = [(t[1], t[2]) for t in triple]
related_triples


In [ ]:
head_spans_found = set()
for pattern in [aliases_patterns_map[als_str] for als_str in aliases[h]]:
    for match in pattern.finditer(sentence):
        start, end = match.span()
        if end ==0 or start >= end:
            continue
        ts = enc.char_to_token(0, start)
        te = enc.char_to_token(0, end - 1)
        if ts is not None and te is not None:
            head_spans_found.add((start, end))
head_spans_found

In [ ]:
" ".join(tokens[0])

In [ ]:
_found_tails = {}
for r, t in related_triples:
    if t not in _found_tails:
        tail_spans = set()
        print(f"r, t: {r, t}")
        print(f"relation aliases: {relations[r]}")
        for pattern in [aliases_patterns_map[als_str] for als_str in aliases[t]]:
            for match in pattern.finditer(sentence):
                start, end = match.span()
                if end == 0 or start >= end:
                    continue
                print(f"\tpattern: {pattern}")
                print(f"\tstart,end: {(start,end)}")
                ts = enc.char_to_token(0, start)
                te = enc.char_to_token(0, end - 1)
                print(f"\t(ts, te): {(ts,te)}")

                if ts is not None and te is not None:
                    tail_spans.add((start, end))
        _found_tails[t] = tail_spans

In [ ]:
import re
pat  = re.compile('(?<!\\w)human(?!\\w)', re.IGNORECASE)
for f in  pat.finditer(sentence):
    print(f.span())

In [ ]:
from collections import defaultdict


len(triples)
triples_per_head = defaultdict(list)
for h,r,t in triples:
    triples_per_head[h].append((r,t))

for h, rs_list in triples_per_head.items():
    r_unq = list(set([ r for r, _ in rs_list]))
    r_all = [r for r, _ in rs_list]
    assert len(r_unq) == len(r_all), f"head {h} has duplicate relations: {r_all}"

In [ ]:
len(descriptions), len(triples)

In [ ]:
from collections import defaultdict
triples_per_head = defaultdict(list)
for h,r,t in triples:
    triples_per_head[h].append((r,t))

for d_id, d in descriptions.items():
    d_triples = triples_per_head[d_id]
    g_triples = golden_triples[d_id]
    assert len(d_triples) == len(g_triples), f"Wrong number of triples d_id: {d_id}"




In [ ]:
triples_per_head["Q135663"], golden_triples["Q135663"]
